# Pew Track Fine-Tuning — Jupyter Notebook

**Read this whole cell before running anything.**

This is the Pew-dataset counterpart to `scripts/trackB_finetune.ipynb` (WVS-7),
restructured to run inside Jupyter on the lab GPU machine. Same safety model as
every notebook/script in this repo: **Preflight → Smoke Test → Real Run**, in
that order, no skipping.

## Zero manual setup — everything is already in this repo

The Pew data (raw CSV + cleaned parquet), the codebook with real verified
question wording (302/304 columns), the 73 screened training items, and the
5-fold split are all **already committed** — nothing to copy onto the lab
machine by hand. `git clone`, install requirements, open this notebook, run
cells top to bottom. That's the whole setup.

## What's different from the WVS Track B notebook

- **Dataset:** Pew Research "Religion in India" (2021), 29,999 respondents
  (vs WVS's 1,692) — national coverage across all of India, not just 8 states.
- **Model:** `openai/gpt-oss-20b` by default, not `gpt-oss-120b` — ~5x smaller
  download (~13GB vs ~60GB), same MoE architecture/chat template, fits
  comfortably on a single lab GPU in 4-bit.

## The one thing that matters most for you: kernel survival

A Jupyter **kernel** keeps running on the server even if you close your browser tab
or lose network — *as long as the Jupyter server process itself stays alive*. If you
started `jupyter lab` / `jupyter notebook` directly in a terminal and that terminal
closes (SSH disconnect, laptop sleep), the server dies **with your training**.

**Before opening this notebook, start Jupyter inside `tmux` on the lab machine:**
```bash
tmux new -s jupyter
jupyter lab --no-browser --ip=0.0.0.0 --port=8888
# leave this running, detach with Ctrl+B then D, reconnect from your laptop's browser
```
This is not optional if you plan to walk away during training — it is the single
biggest cause of "training vanished and I don't know why."

## Recommended way to run this: Kernel → Restart & Run All

Every safety check in this notebook is a **hard `assert`**, not just a printed
warning — so "Run All" is a safe, legitimate way to use this notebook: if
Preflight fails (bad GPU, missing data), or the Smoke Test comes back with a
bad verdict, execution **stops itself automatically** at that cell instead of
continuing into a wasted multi-hour run. You do not have to babysit it and
manually read output between sections — you only need to come back if Jupyter
shows a stopped cell with a red error.

If you'd rather go section by section instead: Setup → Preflight → Smoke Test
→ Real Training Run, in that order, reading each section's output before
moving to the next.

## What survives a crash / disconnect

Everything lives under `pewB_run/` (one folder, see the Setup section for the
full layout) — checkpoints, cached training data, live progress logs, partial
predictions. If the kernel dies mid-training, restart the kernel, re-run the Setup
cells, then re-run the Real Training Run cell — it **auto-resumes from the last
checkpoint**, it does not start over.

## Checking progress without opening the notebook

From a terminal on the lab machine (or another `tmux` pane):
```bash
tail -f pewB_run/logs/train_progress.jsonl
cat pewB_status.json
```

## After training: use the model / build your paper table

```bash
python scripts/pewB_inference.py --model-path pewB_run/model_fold0 \
  --base-model openai/gpt-oss-20b --question-id Q43b --sex Female --religion Hindu

python scripts/pewB_test_panel.py --model-path pewB_run/model_fold0 \
  --base-model openai/gpt-oss-20b --fold 0
```


## 1. Setup — imports

In [ ]:
import json
import logging
import shutil
import sys
import time
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    raise RuntimeError(
        f"Expected to find a 'src' folder next to this notebook at {REPO_ROOT}, "
        "but it's not there. Run this notebook from the repo root "
        "(the folder containing src/, data/, scripts/)."
    )
sys.path.insert(0, str(REPO_ROOT))

from src.config import DATA_PROCESSED, DATA_REFERENCE, RESULTS_DIR
from src.prompts.templates import build_prompt
from src.prompts.verbalize import verbalize_item
from src.prompts.verbalize_pew import load_pew_codebook, verbalize_demographics
from src.inference.prompting import build_answer_instruction, parse_answer_from_text

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    handlers=[logging.StreamHandler()],
)
logger = logging.getLogger("pewB")
print("Imports OK. Repo root:", REPO_ROOT)
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())


## 2. Setup — config

Edit values here if needed. **You should not need to change anything except
possibly `MODEL_NAME`** (see the smaller note below about the two model options).

In [ ]:
CONFIG = {
    "model": "openai/gpt-oss-20b",
    # gpt-oss-20b is the smaller sibling of gpt-oss-120b: same MoE family and
    # chat template, ~13GB native download vs ~60GB, fits easily in 4-bit on
    # a single lab GPU. Swap back to "openai/gpt-oss-120b" here if you want
    # the larger model instead -- everything else in this notebook is
    # unchanged either way.

    "fold": 0,
    "n_items": 15,
    "epochs": 2.0,
    "batch_size": 4,       # gpt-oss-20b is far smaller than the 120B model
    "grad_accum": 8,
    "lr": 2e-4,
    "save_steps": 50,
    "output_dir": "./pewB_run",
    "min_free_disk_gb": 60,
    "min_gpu_mem_gb": 16,   # gpt-oss-20b in 4-bit needs far less than the 120B model's 60GB floor
}

STATUS_PATH = Path("pewB_status.json")  # repo-root, same convention as the .py script
print(json.dumps(CONFIG, indent=2))


### Helper functions — run folder layout, status file, JSONL progress log

In [ ]:
def run_dirs(output_dir: str) -> dict:
    base = Path(output_dir)
    d = {
        "base": base,
        "cache": base / "cache",
        "checkpoints": base / "checkpoints",
        "logs": base / "logs",
        "predictions": base / "predictions",
    }
    for p in d.values():
        p.mkdir(parents=True, exist_ok=True)
    return d


def write_status(phase: str, detail: str = "", extra_log_path: Path = None, **extra):
    status = {"phase": phase, "detail": detail, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"), **extra}
    try:
        STATUS_PATH.write_text(json.dumps(status, indent=2))
    except Exception:
        pass
    if extra_log_path is not None:
        try:
            (extra_log_path / "pewB_status.json").write_text(json.dumps(status, indent=2))
        except Exception:
            pass
    logger.info(f"[STATUS] {phase}: {detail}")


def append_jsonl(path: Path, record: dict):
    try:
        with open(path, "a") as f:
            f.write(json.dumps(record) + "\n")
            f.flush()
    except Exception as e:
        logger.warning(f"Could not append to {path}: {e}")

print("Helpers defined.")


## 3. Preflight — MANDATORY, run this before anything else

Checks CUDA, GPU memory, disk space, that the data files are present, that
Hugging Face Hub is reachable, and actually loads the real model in 4-bit and
runs one real training step. Takes ~5-10 minutes.

**If the cell below prints `PREFLIGHT FAILED`, stop. Do not run the Smoke Test
or Real Run cells. Read the error, fix it (or bring it back to me), then
re-run this cell before continuing.**

In [ ]:
def preflight(cfg: dict) -> bool:
    ok = True
    write_status("preflight", "checking CUDA/GPUs")

    if not torch.cuda.is_available():
        logger.error("PREFLIGHT FAIL: no CUDA device visible. Are you on the GPU node, in the right conda/venv?")
        return False
    n_gpus = torch.cuda.device_count()
    logger.info(f"CUDA OK. {n_gpus} GPU(s) visible:")
    total_mem_gb = 0
    for i in range(n_gpus):
        props = torch.cuda.get_device_properties(i)
        mem_gb = props.total_memory / 1e9
        total_mem_gb += mem_gb
        logger.info(f"  GPU {i}: {props.name}, {mem_gb:.0f} GB")
    if total_mem_gb < cfg["min_gpu_mem_gb"]:
        logger.error(f"PREFLIGHT FAIL: only {total_mem_gb:.0f} GB total GPU memory visible -- too little for {cfg['model']} even in 4-bit (want at least {cfg['min_gpu_mem_gb']} GB).")
        ok = False

    write_status("preflight", "checking disk space")
    free_gb = shutil.disk_usage(".").free / 1e9
    logger.info(f"Free disk space: {free_gb:.0f} GB")
    if free_gb < cfg["min_free_disk_gb"]:
        logger.error(f"PREFLIGHT FAIL: only {free_gb:.0f} GB free, want at least {cfg['min_free_disk_gb']} GB.")
        ok = False

    write_status("preflight", "checking data files present")
    for p in [DATA_PROCESSED / "pew_india_2021.parquet", DATA_PROCESSED / "pew_selected_items.json", DATA_PROCESSED / "pew_folds.json", DATA_REFERENCE / "pew_codebook.json"]:
        if not p.exists():
            logger.error(f"PREFLIGHT FAIL: missing {p} -- this file is committed to the repo, so a missing file usually means an incomplete git clone/checkout (or you're not running from the repo root). Re-clone or re-checkout the pew-dataset-training branch.")
            ok = False
    if not ok:
        return False

    write_status("preflight", "checking selected items are non-empty (codebook wording verified)")
    with open(DATA_PROCESSED / "pew_selected_items.json") as f:
        n_selected = json.load(f)["n_items"]
    if n_selected == 0:
        logger.error(
            "PREFLIGHT FAIL: pew_selected_items.json has 0 items. This means "
            "data/reference/pew_codebook.json still has no verified question "
            "wording -- see src/data/build_pew_codebook.py's docstring (and "
            "this notebook's title cell). Parse Pew's own codebook document, "
            "re-run `python -m src.data.build_pew_codebook --labels <file>`, "
            "then `python -m src.data.select_items_pew`, before continuing."
        )
        return False
    logger.info(f"{n_selected} items selected with verified wording -- OK to proceed.")

    write_status("preflight", "checking Hugging Face Hub reachability")
    try:
        from huggingface_hub import HfApi
        HfApi().model_info(cfg["model"])
        logger.info(f"HF Hub reachable, model repo '{cfg['model']}' found.")
    except Exception as e:
        logger.error(f"PREFLIGHT FAIL: could not reach Hugging Face Hub or find '{cfg['model']}': {e}")
        logger.error("If this is a gated model, run `huggingface-cli login` in a terminal first, then restart the kernel.")
        return False

    write_status("preflight", f"loading {cfg['model']} in 4-bit and running one real train step (slow part, several minutes)")
    try:
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
        from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16,
        )
        tokenizer = AutoTokenizer.from_pretrained(cfg["model"], trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(
            cfg["model"], quantization_config=bnb_config, device_map="auto", trust_remote_code=True,
        )
        model.config.use_cache = False
        model = prepare_model_for_kbit_training(model)
        lora_config = LoraConfig(
            r=16, lora_alpha=32, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
            lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
        )
        model = get_peft_model(model, lora_config)

        dummy_text = "This is a preflight check. " * 40
        inputs = tokenizer(dummy_text, return_tensors="pt", truncation=True, max_length=700).to(model.device)
        labels = inputs["input_ids"].clone()
        out = model(**inputs, labels=labels)
        out.loss.backward()
        model.zero_grad()
        peak_mem_gb = max(torch.cuda.max_memory_allocated(i) for i in range(n_gpus)) / 1e9
        logger.info(f"Model loaded, one train step ran successfully. Peak single-GPU memory: {peak_mem_gb:.1f} GB.")
        del model, out
        torch.cuda.empty_cache()
    except torch.cuda.OutOfMemoryError as e:
        logger.error(f"PREFLIGHT FAIL: OOM loading/stepping {cfg['model']} in 4-bit: {e}")
        logger.error("Try reducing CONFIG['batch_size'], then restart the kernel and re-run.")
        return False
    except Exception as e:
        logger.error(f"PREFLIGHT FAIL: error loading/running {cfg['model']}: {e}")
        logger.error(traceback.format_exc())
        return False

    write_status("preflight", "PASSED -- safe to run the Smoke Test section next")
    logger.info("=" * 60)
    logger.info("PREFLIGHT PASSED.")
    logger.info("=" * 60)
    return True


PREFLIGHT_PASSED = preflight(CONFIG)
assert PREFLIGHT_PASSED, "PREFLIGHT FAILED -- read the error above. Do NOT run the cells below until this passes."
print("\n>>> Preflight passed. Safe to continue to the Smoke Test section. <<<")


## 4. Data pipeline — build (or reuse cached) training examples

In [ ]:
def build_examples(df: pd.DataFrame, respondent_ids: list, selected_items: list, codebook: dict) -> list:
    examples = []
    skipped = 0
    sub = df[df["respondent_id"].isin(respondent_ids)]
    for _, row in sub.iterrows():
        try:
            demo = verbalize_demographics(row, codebook)
        except Exception as e:
            skipped += 1
            logger.warning(f"Skipping respondent {row.get('respondent_id')}: verbalize_demographics failed ({e})")
            continue
        for question_id in selected_items:
            try:
                item = verbalize_item(question_id, codebook)
                true_code = row.get(question_id)
                if pd.isna(true_code):
                    continue
                true_code = int(true_code)
                if true_code not in item["code_to_index"]:
                    continue
                option_labels = [str(c) for c in item["ordinal_values"]]
                prompt = build_prompt(
                    "P2", item["question_text"], item["options_text"], **demo
                ) + build_answer_instruction(option_labels)
                answer = str(true_code)
                examples.append({
                    "text": prompt + "\n\n" + answer,
                    "prompt": prompt,
                    "answer": answer,
                    "respondent_id": int(row["respondent_id"]),
                    "question_id": question_id,
                })
            except Exception as e:
                skipped += 1
                logger.warning(f"Skipping ({row.get('respondent_id')}, {question_id}): {e}")
    if skipped:
        logger.warning(f"Skipped {skipped} (respondent, item) pairs due to errors -- see warnings above.")
    return examples


def load_or_build_examples(df, respondent_ids, selected_items, codebook, cache_path: Path, rebuild: bool = False) -> list:
    if cache_path.exists() and not rebuild:
        examples = []
        with open(cache_path) as f:
            for line in f:
                line = line.strip()
                if line:
                    examples.append(json.loads(line))
        logger.info(f"Loaded {len(examples)} cached training examples from {cache_path} (delete this file to force a rebuild).")
        return examples

    examples = build_examples(df, respondent_ids, selected_items, codebook)
    with open(cache_path, "w") as f:
        for ex in examples:
            f.write(json.dumps(ex) + "\n")
    logger.info(f"Built and cached {len(examples)} training examples to {cache_path}")
    return examples


def sanity_check_examples(examples: list, min_expected: int):
    if len(examples) < min_expected:
        raise RuntimeError(
            f"Only built {len(examples)} training examples, expected at least {min_expected}. "
            f"This smells like a data/config bug, not normal missingness -- STOP and investigate "
            f"rather than burning GPU time on a broken dataset."
        )

print("Data pipeline functions defined.")


### Model loader (shared by smoke test and real run)

In [ ]:
def make_tokenizer(cfg):
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(cfg["model"], trust_remote_code=True, padding_side="right")
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok


def model_loader(cfg):
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16,
    )
    m = AutoModelForCausalLM.from_pretrained(
        cfg["model"], quantization_config=bnb_config, device_map="auto", trust_remote_code=True,
    )
    m.config.use_cache = False
    m = prepare_model_for_kbit_training(m)
    lora_config = LoraConfig(
        r=16, lora_alpha=32, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    )
    m = get_peft_model(m, lora_config)
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    total = sum(p.numel() for p in m.parameters())
    logger.info(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.3f}%)")
    return m

print("Model loader defined.")


### Live progress logging callback (JSONL, one line per training step)

In [ ]:
from transformers import TrainerCallback  # base class -- REQUIRED, see note below

class JsonlProgressCallback(TrainerCallback):
    '''Inherits TrainerCallback so every hook Trainer calls (on_init_end,
    on_train_begin, on_step_begin, etc.) resolves to the base class's no-op
    default except the 3 we override below. A version of this class that does
    NOT inherit from TrainerCallback will crash with AttributeError the
    moment training starts -- Trainer calls ~16 hook methods on every
    registered callback via getattr(), and a plain object only has the ones
    you explicitly wrote.'''

    def __init__(self, jsonl_path: Path, status_extra_dir: Path, total_steps: int):
        self.jsonl_path = jsonl_path
        self.status_extra_dir = status_extra_dir
        self.total_steps = max(total_steps, 1)
        self.t_start = time.time()

    def _eta_minutes(self, step: int) -> float:
        if step <= 0:
            return float("nan")
        elapsed = time.time() - self.t_start
        rate = elapsed / step
        remaining = max(self.total_steps - step, 0)
        return (remaining * rate) / 60.0

    def on_log(self, args, state, control, logs=None, **kwargs):
        logs = logs or {}
        if "loss" not in logs:
            return control
        step = state.global_step
        pct = 100.0 * step / self.total_steps
        eta_min = self._eta_minutes(step)
        record = {
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            "step": step, "total_steps": self.total_steps, "pct_complete": round(pct, 1),
            "epoch": round(logs.get("epoch", state.epoch or 0), 3),
            "loss": logs.get("loss"), "learning_rate": logs.get("learning_rate"),
            "grad_norm": logs.get("grad_norm"),
            "elapsed_min": round((time.time() - self.t_start) / 60.0, 1),
            "eta_min": round(eta_min, 1) if eta_min == eta_min else None,
        }
        append_jsonl(self.jsonl_path, record)
        write_status(
            "training",
            f"step {step}/{self.total_steps} ({pct:.1f}%), epoch {record['epoch']}, loss {record['loss']}, ETA {record['eta_min']} min",
            extra_log_path=self.status_extra_dir,
            step=step, total_steps=self.total_steps, pct_complete=round(pct, 1),
            epoch=record["epoch"], loss=record["loss"], eta_min=record["eta_min"],
        )
        return control

    def on_epoch_end(self, args, state, control, **kwargs):
        append_jsonl(self.jsonl_path, {
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"), "event": "epoch_end",
            "epoch": round(state.epoch or 0, 3), "step": state.global_step,
        })
        logger.info(f"=== Epoch {round(state.epoch or 0, 2)} complete (step {state.global_step}/{self.total_steps}) ===")
        return control

    def on_save(self, args, state, control, **kwargs):
        append_jsonl(self.jsonl_path, {
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"), "event": "checkpoint_saved", "step": state.global_step,
        })
        logger.info(f"Checkpoint saved at step {state.global_step}")
        return control

print("JsonlProgressCallback defined (correctly inherits TrainerCallback).")


## 5. Training function (shared by Smoke Test and Real Run)

Uses `trl.SFTConfig` + `SFTTrainer` with the **current** API (verified against the
actual installed `trl`/`transformers` versions on the machine this notebook was
built on — see the compatibility note below the code if your lab machine resolves
different versions via `pip install`).

Handles OOM by halving batch size and retrying (down to batch size 1), and
auto-resumes from the latest checkpoint in `checkpoints/` if one exists.

**On interrupt (Jupyter "Interrupt Kernel", or an unhandled error):** this saves an
emergency checkpoint before the exception propagates, so a manual stop is not a
lost run either — re-running the training cell resumes from it.

In [ ]:
def find_latest_checkpoint(checkpoints_dir: Path):
    if not checkpoints_dir.exists():
        return None
    checkpoints = sorted(checkpoints_dir.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
    return str(checkpoints[-1]) if checkpoints else None


def train_with_oom_backoff(cfg, train_examples, tokenizer, dirs: dict):
    from datasets import Dataset
    from trl import SFTConfig, SFTTrainer

    batch_size = cfg["batch_size"]
    grad_accum = cfg["grad_accum"]
    last_error = None

    effective_batch = max(batch_size * grad_accum, 1)
    steps_per_epoch = max(len(train_examples) // effective_batch, 1)
    total_steps = int(steps_per_epoch * cfg["epochs"])
    progress_jsonl = dirs["logs"] / "train_progress.jsonl"

    hf_dataset = Dataset.from_list(train_examples)

    while batch_size >= 1:
        try:
            model = model_loader(cfg)
            resume_from = find_latest_checkpoint(dirs["checkpoints"])
            if resume_from:
                logger.info(f"Found existing checkpoint {resume_from} -- resuming, NOT starting over.")

            sft_config = SFTConfig(
                output_dir=str(dirs["checkpoints"]),
                num_train_epochs=cfg["epochs"],
                per_device_train_batch_size=batch_size,
                gradient_accumulation_steps=grad_accum,
                warmup_steps=20,
                learning_rate=cfg["lr"],
                weight_decay=0.01,
                bf16=True,
                logging_steps=10,
                save_steps=cfg["save_steps"],
                save_total_limit=3,
                optim="paged_adamw_32bit",
                seed=42,
                max_grad_norm=1.0,
                remove_unused_columns=False,
                report_to="none",
                packing=False,
                max_length=768,
                dataset_text_field="text",  # matches the "text" key every example dict has
            )
            trainer = SFTTrainer(
                model=model,
                args=sft_config,
                train_dataset=hf_dataset,
                processing_class=tokenizer,  # NOTE: current trl uses processing_class, not tokenizer=
            )
            trainer.add_callback(JsonlProgressCallback(progress_jsonl, dirs["logs"], total_steps))

            write_status(
                "training", f"batch_size={batch_size} grad_accum={grad_accum}",
                extra_log_path=dirs["logs"], n_examples=len(train_examples), total_steps=total_steps,
            )

            try:
                trainer.train(resume_from_checkpoint=resume_from)
            except KeyboardInterrupt:
                logger.warning("Interrupted (Jupyter kernel interrupt) -- saving emergency checkpoint before re-raising.")
                write_status("training", "interrupted, saving emergency checkpoint", extra_log_path=dirs["logs"])
                try:
                    trainer.save_model(str(dirs["checkpoints"] / "emergency_checkpoint"))
                    append_jsonl(progress_jsonl, {"timestamp": time.strftime("%Y-%m-%d %H:%M:%S"), "event": "emergency_checkpoint_saved", "step": trainer.state.global_step})
                    logger.info(f"Emergency checkpoint saved. Re-run this cell to resume from step {trainer.state.global_step}.")
                except Exception:
                    logger.error("Emergency checkpoint save failed too.")
                raise

            return trainer, model

        except torch.cuda.OutOfMemoryError as e:
            last_error = e
            torch.cuda.empty_cache()
            logger.warning(f"OOM at batch_size={batch_size}. Halving batch size and doubling grad_accum, then retrying.")
            append_jsonl(progress_jsonl, {"timestamp": time.strftime("%Y-%m-%d %H:%M:%S"), "event": "oom_backoff", "old_batch_size": batch_size, "new_batch_size": batch_size // 2})
            batch_size //= 2
            grad_accum *= 2
            write_status("training", f"OOM recovery: retrying at batch_size={batch_size}", extra_log_path=dirs["logs"])

    raise RuntimeError(f"Training failed even at batch_size=1: {last_error}")

print("Training function defined (verified against installed trl==%s)." % __import__("trl").__version__)


**Compatibility note:** the cell above targets the *current* `trl`/`transformers`
API as of when this notebook was built (`trl` 1.x — `SFTConfig` with `max_length`
and `dataset_text_field`, `processing_class=` instead of `tokenizer=`). If
`pip install -r scripts/pewB_requirements.txt` on the lab machine resolves a
*different* major version and the training cell errors with something like
`unexpected keyword argument`, that's a version drift, not a logic bug — run this
in a terminal first to check what actually installed, and tell me the exact
versions + error before changing anything yourself:
```bash
python -c "import trl, transformers; print(trl.__version__, transformers.__version__)"
```

## 6. Smoke Test — MANDATORY, run before the real training run

Tiny data (5 train, 5 test respondents, 3 items), full pipeline including a
checkpoint save, so any data/schema/API bug shows up here (~10-20 min) instead of
hours into the real run.

**Read the printed verdict at the end. If it says `REVIEW NEEDED`, stop and look at
the sample predictions before running the real training section.**

In [ ]:
def load_data_for_run(cfg, smoke_test: bool):
    df = pd.read_parquet(DATA_PROCESSED / "pew_india_2021.parquet")
    if "respondent_id" not in df.columns:
        df["respondent_id"] = range(len(df))
    with open(DATA_PROCESSED / "pew_selected_items.json") as f:
        selected_items = json.load(f)["selected_items"][: cfg["n_items"]]
    with open(DATA_PROCESSED / "pew_folds.json") as f:
        folds = json.load(f)["folds"]
    codebook = load_pew_codebook()

    if not selected_items:
        raise RuntimeError(
            "0 selected items -- pew_codebook.json has no verified question "
            "wording yet. Re-run the Preflight cell; it explains exactly "
            "what's missing and how to fix it."
        )

    fold = folds[cfg["fold"]]
    train_ids, test_ids = fold["train"], fold["test"]
    run_tag = f"fold{cfg['fold']}"
    output_dir = cfg["output_dir"]
    min_expected = 1000

    if smoke_test:
        train_ids, test_ids = train_ids[:5], test_ids[:5]
        selected_items = selected_items[:3]
        output_dir = output_dir + "_smoketest"
        run_tag = "smoketest"
        min_expected = 10
        logger.info("SMOKE TEST: 5 train respondents, 5 test respondents, 3 items")

    return df, selected_items, codebook, train_ids, test_ids, run_tag, output_dir, min_expected


def run_pipeline(cfg, smoke_test: bool):
    df, selected_items, codebook, train_ids, test_ids, run_tag, output_dir, min_expected = load_data_for_run(cfg, smoke_test)
    dirs = run_dirs(output_dir)

    run_log_handler = logging.FileHandler(str(dirs["logs"] / "pewB_run.log"))
    run_log_handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
    logging.getLogger().addHandler(run_log_handler)

    t_run_start = time.time()
    logger.info(f"Fold {cfg['fold']}: {len(train_ids)} train, {len(test_ids)} test, {len(selected_items)} items")
    logger.info(f"All output for this run: {dirs['base'].resolve()}")
    write_status("building_data", "constructing training examples", extra_log_path=dirs["logs"])

    cache_path = dirs["cache"] / f"train_examples_{run_tag}.jsonl"
    train_examples = load_or_build_examples(df, train_ids, selected_items, codebook, cache_path)
    sanity_check_examples(train_examples, min_expected)
    logger.info(f"Built {len(train_examples)} training examples")

    tokenizer = make_tokenizer(cfg)

    write_status("training", "starting", extra_log_path=dirs["logs"])
    t0 = time.time()
    trainer, model = train_with_oom_backoff(cfg, train_examples, tokenizer, dirs)
    train_minutes = (time.time() - t0) / 60
    logger.info(f"Training done in {train_minutes:.1f} min")

    model_path = str(dirs["base"] / f"model_{run_tag}")
    trainer.model.save_pretrained(model_path)
    tokenizer.save_pretrained(model_path)
    logger.info(f"Saved fine-tuned model to {model_path}")
    write_status("training_complete", f"saved to {model_path}", extra_log_path=dirs["logs"], train_minutes=round(train_minutes, 1))

    # ---- Out-of-fold prediction on test respondents ----
    write_status("evaluating", "running out-of-fold inference", extra_log_path=dirs["logs"])
    model.eval()
    rows = []
    test_sub = df[df["respondent_id"].isin(test_ids)]
    t_eval0 = time.time()
    for i, (_, row) in enumerate(test_sub.iterrows()):
        try:
            demo = verbalize_demographics(row, codebook)
        except Exception as e:
            logger.warning(f"Skipping test respondent {row.get('respondent_id')}: {e}")
            continue
        for question_id in selected_items:
            try:
                item = verbalize_item(question_id, codebook)
                true_code = row.get(question_id)
                if pd.isna(true_code):
                    continue
                true_code = int(true_code)
                if true_code not in item["code_to_index"]:
                    continue
                option_labels = [str(c) for c in item["ordinal_values"]]
                prompt = build_prompt("P2", item["question_text"], item["options_text"], **demo) + build_answer_instruction(option_labels)

                inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=700).to(model.device)
                with torch.no_grad():
                    out = model.generate(**inputs, max_new_tokens=10, do_sample=False, pad_token_id=tokenizer.pad_token_id)
                gen_text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
                pred_code = parse_answer_from_text(gen_text, option_labels)
                pred_code_idx = item["code_to_index"].get(int(pred_code)) if pred_code is not None else None

                idx_to_label = dict(zip(item["ordinal_values"], (l.split(". ", 1)[1] for l in item["options_text"].splitlines())))
                rows.append({
                    "respondent_id": int(row["respondent_id"]), "question_id": question_id,
                    "question_text": item["question_text"], "condition": "P2",
                    "model": f"pewB_{cfg['model'].replace('/', '_')}_fold{cfg['fold']}",
                    "true_code": true_code, "true_label": idx_to_label.get(true_code),
                    "true_code_idx": item["code_to_index"][true_code],
                    "pred_code": int(pred_code) if pred_code is not None else None,
                    "pred_label": idx_to_label.get(int(pred_code)) if pred_code is not None else None,
                    "pred_code_idx": pred_code_idx, "pred_raw_text": gen_text, "refusal": pred_code is None,
                })
            except Exception as e:
                logger.warning(f"Prediction failed for ({row.get('respondent_id')}, {question_id}): {e}")

        if (i + 1) % 20 == 0 or (i + 1) == len(test_sub):
            elapsed = time.time() - t_eval0
            partial_acc = None
            if rows:
                done = pd.DataFrame(rows)
                partial_acc = round(float((done["pred_code_idx"] == done["true_code_idx"]).mean()), 4)
            write_status("evaluating", f"{i+1}/{len(test_sub)} respondents, {len(rows)} predictions, running accuracy {partial_acc}",
                         extra_log_path=dirs["logs"], respondents_done=i + 1, respondents_total=len(test_sub), running_accuracy=partial_acc)
            pd.DataFrame(rows).to_parquet(dirs["predictions"] / "partial_predictions.parquet")

    pred_df = pd.DataFrame(rows)
    out_name = f"pewB_{cfg['model'].replace('/', '_')}_fold{cfg['fold']}_P2.parquet"
    out_path = Path(RESULTS_DIR) / "predictions" / out_name
    out_path.parent.mkdir(parents=True, exist_ok=True)
    pred_df.to_parquet(out_path)
    pred_df.to_parquet(dirs["predictions"] / out_name)

    n = len(pred_df)
    n_answered = pred_df["pred_code_idx"].notna().sum() if n else 0
    acc = (pred_df["pred_code_idx"] == pred_df["true_code_idx"]).mean() if n else float("nan")
    summary = {
        "model": cfg["model"], "fold": cfg["fold"], "n_items": len(selected_items),
        "n_predictions": n, "n_answered": int(n_answered),
        "refusal_rate": round(1 - n_answered / n, 4) if n else None,
        "raw_accuracy": round(float(acc), 4) if n else None,
        "train_minutes": round(train_minutes, 1),
        "total_run_minutes": round((time.time() - t_run_start) / 60, 1),
        "output_path": str(out_path), "run_folder": str(dirs["base"].resolve()),
    }
    write_status("complete", "run finished successfully", extra_log_path=dirs["logs"], **summary)

    if smoke_test:
        preview = pred_df.head(10)[["respondent_id", "question_id", "true_code", "pred_code", "pred_raw_text", "refusal"]].to_dict(orient="records") if n else []
        smoke_result = {
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            "n_train_examples": len(train_examples), "n_predictions": n,
            "refusal_rate": summary["refusal_rate"], "raw_accuracy_on_tiny_sample": summary["raw_accuracy"],
            "note": "Accuracy on this tiny sample is NOT meaningful (n too small). What matters: did this complete without errors, and do pred_raw_text values look like clean digits, not garbage/refusals.",
            "sample_predictions": preview,
            "verdict": "PASS -- safe to proceed to the real run" if (summary["refusal_rate"] is not None and summary["refusal_rate"] < 0.5) else "REVIEW NEEDED -- high refusal/unparsed rate, read sample_predictions before proceeding",
        }
        smoke_path = dirs["logs"] / "smoke_test_result.json"
        smoke_path.write_text(json.dumps(smoke_result, indent=2))
        logger.info(f"Smoke test summary written to {smoke_path}")
        print("\n" + "=" * 60)
        print("SMOKE TEST VERDICT:", smoke_result["verdict"])
        print("=" * 60)
        for p in preview:
            print(p)
        # Surfaced on the returned dict (not just printed/saved to file) so the
        # Real Run cell can assert on it directly -- this is what makes
        # Kernel > Restart & Run All a safe, unattended way to use this
        # notebook: a bad smoke test stops execution there instead of silently
        # continuing into a wasted multi-hour run.
        summary["smoke_verdict"] = smoke_result["verdict"]

    print("\n" + "=" * 60)
    print("RUN SUMMARY")
    print(json.dumps(summary, indent=2))
    print("=" * 60)
    return summary

print("run_pipeline() defined.")


In [ ]:
# Run the smoke test now. ~10-20 minutes.
smoke_summary = run_pipeline(CONFIG, smoke_test=True)


**Stop here and read the verdict printed above (and in
`pewB_run_smoketest/logs/smoke_test_result.json`) before continuing.**
If it says `REVIEW NEEDED`, do not run the real training cell -- investigate first.


## 7. Real Training Run — the multi-hour cell

Only run this after Preflight passed and the Smoke Test verdict was `PASS`.

This is safe to walk away from (see the top of this notebook for kernel-survival
setup via `tmux`). If interrupted or the kernel dies, re-run the **Setup**
cells above, then re-run this cell — it resumes from the last checkpoint under
`trackB_run/checkpoints/`, it does not restart.

In [ ]:
# THE REAL RUN. gpt-oss-20b trains considerably faster than the 120B model --
# expect roughly 1-3 hours total (training + evaluation) on a single modern GPU,
# but always trust the live ETA in train_progress.jsonl over this comment.
assert "PASS" in smoke_summary.get("smoke_verdict", ""), (
    f"Smoke test did not pass (verdict: {smoke_summary.get('smoke_verdict')!r}) -- "
    "refusing to start the real run. Read pewB_run_smoketest/logs/smoke_test_result.json, "
    "fix the underlying issue, and re-run the Smoke Test cell before retrying this one."
)
real_run_summary = run_pipeline(CONFIG, smoke_test=False)


## 8. When it's done

Bring back (same USB stick, or however you move files off the lab machine):

```
pewB_run/                                      <- whole folder, or at minimum:
pewB_run/logs/train_progress.jsonl
pewB_run/logs/pewB_run.log
pewB_status.json                               <- repo root
results/predictions/pewB_<model>_fold0_P2.parquet
```

Paste the `real_run_summary` dict printed above back into the chat -- it has the
final accuracy, refusal rate, and file paths in one place.

To answer NEW questions with the trained model without retraining, or to build
a paper-ready results table from a fixed panel of held-out respondents, see
`scripts/pewB_inference.py` and `scripts/pewB_test_panel.py`.
